# 02 — SuRVoS-Style 2.5D Volumetric Propagation & Tri-Plane Epistemic Certainty

In synchrotron XCT + XRD-CT (e.g., Diamond DIAD `Zn13X`) or FIB-SEM + sparse EDS tomography, acquiring a dense 3D structural volume is fast, whereas chemical/diffraction scans are often collected on only a handful of $z$-slices.

`VolumetricFuser` bridges 2D ViT backbones to 3D volumes at zero 3D retraining cost via:
1. **Adjacent-Slice RGB Packing (`2.5d_slab`)**: packs `(z - Δz, z, z + Δz)` into `(R, G, B)`.
2. **Sparse-Slice Probe Fitting (`fit_sparse_slices`)**: trains the chemical fusion head on the few annotated $z$-slices and propagates across the entire 3D tomogram.
3. **Orthogonal Tri-Plane Aggregation (`triplane`)**: fuses `XY + XZ + YZ` predictions and uses inter-plane variance as a 3D epistemic uncertainty map.

In [ ]:
import matplotlib.pyplot as plt
import torch

from x_fuse import (
    CorrelativeFuser,
    VolumetricFuser,
    compute_segmentation_metrics,
    generate_correlative_phantom_3d,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE = "dinov2_vits14_reg" if DEVICE == "cuda" else "timm:vit_tiny_patch16_224"
PRETRAINED = DEVICE == "cuda"

## 1. Create a 3D Multiphase Tomogram & Extract Only 3 Sparse Chemical $z$-Slices

In [ ]:
phantom_3d = generate_correlative_phantom_3d(
    shape=(32, 64, 64),
    downsample_factor=4,
    seed=123,
)

# Simulate acquiring coarse chemical maps on only 3 sparse XY slices out of 32!
sparse_z_indices = [4, 16, 27]
sparse_guides = {
    z: phantom_3d.gt_phase_channels_3d[:, z].reshape(2, 16, 4, 16, 4).mean(axis=(2, 4))
    for z in sparse_z_indices
}
print("Full HR volume shape (Z, Y, X):", phantom_3d.hr_volume.shape)
print("Sparse chemical slices acquired at z =", list(sparse_guides.keys()))

## 2. Fit on Sparse Slices & Propagate via 2.5D Orthogonal Tri-Plane Fusion

In [ ]:
stride = 7 if "14" in BACKBONE else 8
base_fuser = CorrelativeFuser(
    dino_name=BACKBONE,
    xrd_fuse_method="learned_gating",
    stride=stride,
    dtype=torch.float32,
    loss_fn="forward_psf_bce",
    guide_channels=2,
    interpolation_mode="jbu",
    pretrained=PRETRAINED,
    lr=1e-2,
    device=DEVICE,
)

vol_fuser = VolumetricFuser(base_fuser)
losses = vol_fuser.fit_sparse_slices(phantom_3d.hr_volume, sparse_guides, epochs=15, delta=1)
vol_report = vol_fuser.predict_volume(
    phantom_3d.hr_volume,
    mode="triplane",
    delta=1,
    sparse_guides=sparse_guides,
    pred_mode="multiclass",
)

pred_3d = vol_report.hard_mask.numpy()
metrics_3d = compute_segmentation_metrics(pred_3d, phantom_3d.gt_labels_3d)
print("3D Volumetric mIoU:       ", round(metrics_3d["iou"], 4))
print("z-Slice Consistency Score:", round(vol_report.z_slice_consistency, 4))
print("Mean Tri-Plane Variance:  ", round(float(vol_report.triplane_variance.mean()), 6))

## 3. Inspect an Unseen Interior $z$-Slice (`z = 11`)

In [ ]:
unseen_z = 11
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), constrained_layout=True)
axes[0].imshow(phantom_3d.hr_volume[unseen_z], cmap="gray")
axes[0].set_title(f"HR Tomogram (z={unseen_z}, unseen)")
axes[1].imshow(phantom_3d.gt_labels_3d[unseen_z], cmap="viridis", vmin=0, vmax=2)
axes[1].set_title("3D Ground Truth")
axes[2].imshow(pred_3d[unseen_z], cmap="viridis", vmin=0, vmax=2)
axes[2].set_title("Tri-Plane Propagated Mask")
axes[3].imshow(vol_report.triplane_variance[unseen_z], cmap="inferno")
axes[3].set_title("Tri-Plane Epistemic Variance")
for ax in axes:
    ax.axis("off")
plt.show()